# FastWAM LIBERO evaluation

This notebook runs FastWAM through the repo's LIBERO evaluator and reports the task success rate.

The default matches FastWAM's single-task evaluation budget: task 0 of libero_spatial, 50 full-length episodes, 30 initial no-op steps, and replanning every 10 actions. This gives a task-specific number, not the full LIBERO benchmark average. Compare it with FastWAM's result for the same suite, task, checkpoint, and settings.

Run the cells top to bottom once on the H100 with internet access. The checkpoint is on Hugging Face and the Wan components are on ModelScope. Both are public, so no login is needed.

In [ ]:
!git clone --branch LIBERO --single-branch https://github.com/Neural-Representation-Dynamics-Lab/wam_interp.git wam_interp

from pathlib import Path

REPO_PATH = Path("wam_interp").resolve()
CHECKPOINTS_PATH = REPO_PATH / "checkpoints" / "fastwam_release"
CHECKPOINTS_PATH.mkdir(exist_ok=True, parents=True)
print(f"Repo: {REPO_PATH}")

## Install the repo and LIBERO

The repo extra installs FastWAM and the compatible MuJoCo/robosuite dependencies. LIBERO's source checkout supplies its tasks and simulator assets, so the notebook installs that source without its old pinned dependencies.

In [ ]:
%pip install -q -e "./wam_interp[all]"

!mkdir -p wam_interp/third_party
!git clone --depth 1 --no-checkout --branch master https://github.com/Lifelong-Robot-Learning/LIBERO.git wam_interp/third_party/LIBERO
!git -C wam_interp/third_party/LIBERO fetch --depth 1 origin 8f1084e3132a39270c3a13ebe37270a43ece2a01
!git -C wam_interp/third_party/LIBERO checkout --detach 8f1084e3132a39270c3a13ebe37270a43ece2a01
%pip install -q --no-deps -e wam_interp/third_party/LIBERO --config-settings editable_mode=compat

import os
import yaml

os.environ.setdefault("MUJOCO_GL", "egl")

LIBERO_ROOT = REPO_PATH / "third_party" / "LIBERO" / "libero" / "libero"
LIBERO_CONFIG_DIR = REPO_PATH / "libero_config"
LIBERO_CONFIG_DIR.mkdir(exist_ok=True, parents=True)
LIBERO_DATASETS_DIR = REPO_PATH / "libero_datasets_empty"
LIBERO_DATASETS_DIR.mkdir(exist_ok=True, parents=True)

libero_paths = {
    "benchmark_root": str(LIBERO_ROOT),
    "bddl_files": str(LIBERO_ROOT / "bddl_files"),
    "init_states": str(LIBERO_ROOT / "init_files"),
    "datasets": str(LIBERO_DATASETS_DIR),
    "assets": str(LIBERO_ROOT / "assets"),
}
(LIBERO_CONFIG_DIR / "config.yaml").write_text(yaml.safe_dump(libero_paths))
os.environ["LIBERO_CONFIG_PATH"] = str(LIBERO_CONFIG_DIR)
print("Installed LIBERO and wrote its path config.")

## Check the simulator before loading FastWAM

This quick reset and camera check catches LIBERO path or rendering problems before downloading and loading the model.

In [ ]:
from libero.libero import benchmark, get_libero_path
from wam_interp.evaluations.LIBERO.environment import (
    create_libero_environment,
    initial_states_for_task,
    reset_to_initial_state,
)

suite = benchmark.get_benchmark_dict()["libero_spatial"]()
task = suite.get_task(0)
initial_state = initial_states_for_task(suite, 0)[0]
bddl = Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
assert bddl.is_file(), f"Missing LIBERO task file: {bddl}"

env = create_libero_environment(task, resolution=256, seed=42)
try:
    observation = reset_to_initial_state(
        env, initial_state, instruction=str(task.language), metadata={}
    )
    assert observation.cameras["agentview"].shape == (256, 256, 3)
    assert observation.cameras["wrist"].shape == (256, 256, 3)
finally:
    env.close()

print("LIBERO reset and both 256x256 camera renders work.")

## Load the checkpoint and LIBERO adapter

This uses the released FastWAM checkpoint and matching normalization statistics. The LIBERO adapter applies the model's required camera/state preprocessing and converts predicted actions to LIBERO commands.

In [ ]:
import os
from huggingface_hub import hf_hub_download
from wam_interp.models.fastwam.loader import compose_fastwam_config, load_fastwam
from wam_interp.models.fastwam.libero_adapter import FastWAMLiberoAdapter

CHECKPOINT = CHECKPOINTS_PATH / "libero_uncond_2cam224.pt"
STATS = CHECKPOINTS_PATH / "libero_uncond_2cam224_dataset_stats.json"
for filename in (CHECKPOINT.name, STATS.name):
    hf_hub_download(
        repo_id="yuanty/fastwam",
        filename=filename,
        local_dir=CHECKPOINTS_PATH,
    )

WAN_ASSETS_PATH = REPO_PATH / "checkpoints" / "wan_assets"
WAN_ASSETS_PATH.mkdir(exist_ok=True, parents=True)
os.environ["DIFFSYNTH_MODEL_BASE_PATH"] = str(WAN_ASSETS_PATH)

FASTWAM_TASK = "libero_uncond_2cam224_1e-4"
fastwam_cfg = compose_fastwam_config(FASTWAM_TASK, repository_root=REPO_PATH)
raw_model = load_fastwam(
    fastwam_cfg, checkpoint=CHECKPOINT, device="cuda", dtype="float16"
)
adapter = FastWAMLiberoAdapter(
    raw_model,
    data_config=fastwam_cfg.data.train,
    dataset_stats=STATS,
)
print("FastWAM and its LIBERO adapter are ready.")

## Run the evaluation

This uses the suite, task, episode count, warm-up, replanning interval, action horizon, and inference-step settings from the repo's FastWAM config. Send back the printed success rate and the JSON result file.

In [ ]:
from IPython.display import FileLink, display
from wam_interp.evaluations.LIBERO import LiberoEvalConfig, evaluate_libero

suite_name = str(fastwam_cfg.EVALUATION.task_suite_name)
task_id = int(fastwam_cfg.EVALUATION.task_id)
num_trials = int(fastwam_cfg.EVALUATION.num_trials)
results_dir = REPO_PATH / "evaluate_results"

config = LiberoEvalConfig(
    task_suite_name=suite_name,
    task_ids=(task_id,),
    num_trials=num_trials,
    num_steps_wait=int(fastwam_cfg.EVALUATION.num_steps_wait),
    replan_steps=int(fastwam_cfg.EVALUATION.replan_steps),
    action_horizon=int(fastwam_cfg.data.train.num_frames) - 1,
    seed=int(fastwam_cfg.seed),
    output_dir=results_dir,
    inference_options={
        "condition": "first_frame",
        "num_inference_steps": int(fastwam_cfg.EVALUATION.num_inference_steps),
    },
    run_metadata={
        "checkpoint": str(CHECKPOINT),
        "dataset_stats": str(STATS),
        "fastwam_task_config": FASTWAM_TASK,
    },
)

result = evaluate_libero(model=adapter, config=config)

print(f"LIBERO success rate: {result.successes}/{result.episodes} = {result.success_rate:.1%}")
print(f"Suite: {suite_name}; task: {task_id}; trials: {num_trials}")
print(f"Result file: {result.output_path}")
display(FileLink(result.output_path))